In [ ]:
import random
import numpy as np
from src.analyse_equations.analyse_equations import add_all_data_error, print_units_of_one_equation
from src.analyse_equations.analyse_equations import set_pandas_options, create_error_table
from src.analyse_equations.analyse_equations import add_n_fold_error

from src.analyse_equations.add_information_to_equations import add_proposed_equations

from src.analyse_equations.plot_error_per_system import plot_error_per_system
from src.analyse_equations.plot_predictions_of_one_equation import plot_prediction
from src.analyse_equations.utils import save_proposed_equation, get_data_folds, load_proposed_equations

from src.config.config_equations_for_each_dataset import ConfigEquationDiscovery
from src.equation_discovery.evaluate_equation import map_equation_to_syntax_tree

from src.preprocess_data.preprocess_data import prepare_dataset, get_unit_dict
from src.config.config_analyse_equations import ConfigPlotBestEquation
from src.config.config_load_dataset import ConfigLoadData
from src.SyntaxTree.src.syntax_tree.config_syntax_tree import ConfigSyntaxTree
from src.config.config_hyperparameter import ConfigHyperparameter
import logging
from src.analyse_equations.create_constant_table import create_constant_table
from src.analyse_equations.plot_abs_difference_between_equation import abs_difference_between_equation
from src.analyse_equations.plot_histogram_for_features import histogram_for_features
from src.analyse_equations.plot_error_per_system import heatmap_error_per_system, save_system_error_heatmap
from src.analyse_equations.create_constant_table import save_constant_table
from src.analyse_equations.example_evaluation import save_example_evaluation_dict, get_example_evaluation_dict
from pathlib import Path
from src.analyse_equations.analyse_equations import use_same_exponent_in_column
from src.utils.save_tables import replace_for_latex
from src.utils.save_tables import equation_to_latex
from src.analyse_equations.analyse_equations import create_error_table
from src.analyse_equations.analyse_equations import mean_per_liquid_surface

# Configs

In [ ]:
logger = logging.getLogger(__name__)
set_pandas_options()

parser = ConfigHyperparameter.arguments_parser()
parser = ConfigLoadData.arguments_parser(parser)
parser = ConfigEquationDiscovery.arguments_parser(parser)
parser = ConfigPlotBestEquation.arguments_parser(parser)
parser = ConfigSyntaxTree.arguments_parser(parser)
args, unknown = parser.parse_known_args()
random.seed(args.seed)
np.random.seed(args.seed)
args.save_path = args.ROOT_DIR / (f'results/'
                                  f'{Path(*Path(args.path_to_datasets).parts[1:])}'
                                  f'/{args.exp_name}_ipynb')
args.save_path.mkdir(parents=True, exist_ok=True)
args.unit_dict = get_unit_dict(args)
args.unit_dict['y'] = args.unit_dict[args.target]
args.unit_dimension = 5
measurement_error_dic = {
    'drop_length': 4.2e-5,  # m
    'adv': np.rad2deg(3),
    'rec': np.rad2deg(3),
    'mid': np.rad2deg(3),
    'avg_vel': 5e-3,
    'width': 5e-5,
    'y_center': 5e-5,
    'static_adv': np.rad2deg(3),
    'static_rec': np.rad2deg(3)
}
logging.basicConfig(level=logging.INFO)

print(f"results are saved to {args.save_path}")


# Results to load

In [ ]:
proposed_equations = load_proposed_equations(args)
add_proposed_equations(args, proposed_equations)

#files_test, files_train = get_train_test_files(args, proposed_equations)
folds_dict = get_data_folds(args, proposed_equations)
all_files = []
for excel_name, files in folds_dict.items():
    all_files.extend(files)

# Calculate the mean for each liquid surface combination

In [ ]:
mean_per_liquid_surface(all_files, args)

# Cross Validation

In [ ]:
filtered_dfs_test, filtered_dfs_train, tree = add_n_fold_error(args, folds_dict, measurement_error_dic, proposed_equations)


# Calculate Error over all data

In [ ]:
all_data_dfs = prepare_dataset(args, all_files)
add_all_data_error(all_data_dfs, args, proposed_equations)
save_proposed_equation(args, folds_dict, proposed_equations)

# Example Evaluation

In [ ]:
num_constants = 1

example_evaluation_dict = get_example_evaluation_dict(all_data_dfs, args,
                                                      proposed_equations,
                                                      num_constants)
save_example_evaluation_dict(args, example_evaluation_dict, logger)


# Create error table

## Absolute Error

In [ ]:
num_constants = 1
df_error = create_error_table(args, num_constants, proposed_equations, metric='error')
df_error

## Mean squared error

In [ ]:
indices_best_equations = list(df_error.index)
num_constants = 1
df_error_mse = create_error_table(args, num_constants, proposed_equations, metric='error_mse')
df_error_mse

## Relative Error

In [ ]:
num_constants = 1
df_err_rel = create_error_table(args, num_constants, proposed_equations, metric='err_rel')
df_err_rel

## Error in Percent

In [ ]:
num_constants = 1
df_percent_error = create_error_table(args, num_constants, proposed_equations, metric='err_percent')
df_percent_error

## $R^2$ Error

In [ ]:
num_constants = 1
df_r2_error = create_error_table(args, num_constants, proposed_equations, metric='err_r2', ascending=False)
df_r2_error

## Combine MAE and R2 table


In [ ]:
merged_df = df_error.merge(df_r2_error, on='id',  suffixes=('', '_drop'))
merged_df = merged_df.loc[:, ~merged_df.columns.str.endswith('_drop')]
merged_df = merged_df.drop(columns=['fold mean train error', 'id', 'fold mean train err_r2', 'equation'])

merged_df = use_same_exponent_in_column(merged_df, 'fold mean test error',
                                        exponent=-6)
merged_df = use_same_exponent_in_column(merged_df, 'train all error error',
                                        exponent=-6)
merged_df = use_same_exponent_in_column(merged_df, 'calc. std',
                                        exponent=-6)
merged_df['fold mean test err_r2'] = merged_df['fold mean test err_r2'].astype(float).round(2)
merged_df['train all error err_r2'] = merged_df['train all error err_r2'].astype(float).round(2)
merged_df['infix'] = merged_df.apply(lambda row: equation_to_latex(args, row['infix']), axis=1)
print(merged_df.columns)
merged_df = merged_df[['infix', 'fold mean test error * 10^-6', 'train all error error * 10^-6', 'rank_error', 'fold mean test err_r2', 'train all error err_r2', 'rank_err_r2'
    , 'calc. std * 10^-6'
    ]]
merged_df.set_index('infix', inplace=True)
merged_latex = replace_for_latex(merged_df)
save_path = args.save_path / f'table_with_equations_MAE_R2_combined_long.tex'
with open(save_path, "w") as text_file:
    text_file.write(merged_latex)
logger.info(f"table with errors saved @{save_path}")
merged_df

# Create heat map local

## Absolute Error

In [ ]:
import pandas as pd

indices = indices_best_equations[: 4] + [13] + list(df_r2_error.index) [:3] + [5,6,8,22]
metric = 'error'
pd_dict = heatmap_error_per_system(
    all_data_dfs,
    args,
    df_error,
    proposed_equations,
    indices,
    metric
)
pd.DataFrame(pd_dict)
save_system_error_heatmap(args, pd_dict, metric=metric, multiplier_error=10**6, fmt_error = '.3', round_to_digits=2)

## Mean squared Error

In [ ]:
indices = indices_best_equations[: 4]  + list(df_error_mse.index) [:3]+ [5,6,21]
metric = 'error_mse'
pd_dict = heatmap_error_per_system(
    all_data_dfs,
    args,
    df_error_mse,
    proposed_equations,
    indices,
    metric
)
save_system_error_heatmap(args, pd_dict, metric=metric,  multiplier_error=10**12, fmt_error = '.3')#, round_to_digits=2)
pd.DataFrame(pd_dict)

## Relative Error

In [ ]:
indices = indices_best_equations[: 4]  + list(df_r2_error.index) [:3] + [5,6,21]
metric = 'err_rel'
pd_dict = heatmap_error_per_system(
    all_data_dfs,
    args,
    df_err_rel,
    proposed_equations,
    indices,
    metric
)
save_system_error_heatmap(args, pd_dict, metric=metric,  multiplier_error=1, fmt_error = '.3', round_to_digits=2)
pd.DataFrame(pd_dict)

## Error in Percent

In [ ]:

#indices = indices_best_equations[: 4]+ list(df_percent_error.index) [:4] + [5,6,21]
indices = indices_best_equations[: 4]  + list(df_r2_error.index) [:3] + [5,6,21]
metric = 'err_percent'

pd_dict = heatmap_error_per_system(
    all_data_dfs,
    args,
    df_percent_error,
    proposed_equations,
    indices,
    metric
)

save_system_error_heatmap(args, pd_dict, metric=metric,  multiplier_error=10**-2, fmt_error = '.3', round_to_digits=2)
pd.DataFrame(pd_dict)

## $R^2$ Error

In [ ]:
indices = indices_best_equations[: 4] + [13] + list(df_r2_error.index) [:3] + [5,6,8,22]
metric = 'err_r2'

pd_dict = heatmap_error_per_system(
    all_data_dfs,
    args,
    df_r2_error,
    proposed_equations,
    indices,
    metric
)

save_system_error_heatmap(args, pd_dict, metric=metric,fmt_error = '.4', fmt_const='.2',
                          round_to_digits=2, reverse_color_map=True, kwags={'vmin':-5, 'vmax':1})
pd.DataFrame(pd_dict)

# Create constant table

In [ ]:
index =  indices_best_equations[0]

pd_constants = create_constant_table(
    all_data_dfs,
    args,
    df_error.loc[index].loc['equation'],
    proposed_equations
)
save_constant_table(args, logger, pd_constants)
pd_constants['measured_c1'] = pd_constants['friction_coef'] * pd_constants['viscosity']
pd_constants
#print(df_error.loc[index].loc['equation'])

## Print Constants

In [ ]:
proposed_equations[df_error.loc[8].loc['equation']]

# Uncertainty one equation

In [ ]:
import json

index = indices_best_equations[1]
equation = proposed_equations[df_error.loc[index].loc['equation']]
print(json.dumps(equation['error_propagation'], sort_keys=False, indent=4))

# Print units

In [ ]:
index =  indices_best_equations[0]
print_units_of_one_equation(args, df_error, index, proposed_equations)

# Plot prediction

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from src.analyse_equations.plot_predictions_of_one_equation import get_short_and_sorted_df

for index in indices_best_equations:
    equation = proposed_equations[df_error.loc[index].loc['equation']]
    tree = map_equation_to_syntax_tree(args, df_error.loc[index].loc['equation'], infix=False, catch_exceptions=False)
    tree.constants_in_tree = equation['all_data']['train']['constants']
    plot_prediction(args, filtered_dfs_test, filtered_dfs_train, tree, equation_id=str(index))